ETL PIPELINE


In [14]:
import requests
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("ETL Pipeline").getOrCreate()

url = "https://dummyjson.com/users"
response = requests.get(url)
data = response.json()



In [11]:
users = data["users"]

flat_users = []

for user in users:
    flat_users.append({
        "id": user["id"],
        "firstName": user["firstName"],
        "lastName": user["lastName"],
        "age": user["age"],
        "email": user["email"],
        "city": user["address"]["city"],
        "state": user["address"]["state"]
    })

df = spark.createDataFrame(flat_users)

df.show()

+---+------------+--------------------+---------+---+---------+-------------+
|age|        city|               email|firstName| id| lastName|        state|
+---+------------+--------------------+---------+---+---------+-------------+
| 29|     Phoenix|emily.johnson@x.d...|    Emily|  1|  Johnson|  Mississippi|
| 36|     Houston|michael.williams@...|  Michael|  2| Williams|      Alabama|
| 43|  Washington|sophia.brown@x.du...|   Sophia|  3|    Brown|      Alabama|
| 46|     Seattle|james.davis@x.dum...|    James|  4|    Davis| Pennsylvania|
| 31|Jacksonville|emma.miller@x.dum...|     Emma|  5|   Miller|     Colorado|
| 23|  Fort Worth|olivia.wilson@x.d...|   Olivia|  6|   Wilson|    Tennessee|
| 39|Indianapolis|alexander.jones@x...|Alexander|  7|    Jones|     Delaware|
| 28|  Fort Worth|ava.taylor@x.dumm...|      Ava|  8|   Taylor| Rhode Island|
| 34| San Antonio|ethan.martinez@x....|    Ethan|  9| Martinez|    Louisiana|
| 32|    New York|isabella.anderson...| Isabella| 10| Anderson| 

In [12]:
df.printSchema()

root
 |-- age: long (nullable = true)
 |-- city: string (nullable = true)
 |-- email: string (nullable = true)
 |-- firstName: string (nullable = true)
 |-- id: long (nullable = true)
 |-- lastName: string (nullable = true)
 |-- state: string (nullable = true)



In [13]:
df.describe().show()

+-------+-----------------+----------+--------------------+---------+-----------------+--------+-------+
|summary|              age|      city|               email|firstName|               id|lastName|  state|
+-------+-----------------+----------+--------------------+---------+-----------------+--------+-------+
|  count|               30|        30|                  30|       30|               30|      30|     30|
|   mean|33.56666666666667|      NULL|                NULL|     NULL|             15.5|    NULL|   NULL|
| stddev|5.721606558406075|      NULL|                NULL|     NULL|8.803408430829505|    NULL|   NULL|
|    min|               23|   Chicago|abigail.rivera@x....|  Abigail|                1|Anderson|Alabama|
|    max|               46|Washington|william.gonzalez@...|  William|               30|  Wright|Wyoming|
+-------+-----------------+----------+--------------------+---------+-----------------+--------+-------+



Data Cleaning


In [15]:
clean_df = (
    df
    .dropDuplicates()
    .withColumn('firstName',trim(col('firstName')))
    .withColumn('email',lower(col('email')))
)

In [16]:
clean_df = clean_df.fillna({
    "age": 0
})

In [17]:
clean_df = clean_df.dropna()

In [18]:
clean_df = clean_df.withColumn(
    "FullName",
    concat_ws(" ",
              col("firstName"),
              col("lastName"))
)

In [19]:
clean_df = clean_df.withColumn(
    "AgeGroup",
    when(col("age") >= 60, "Senior")
    .when(col("age") >= 18, "Adult")
    .otherwise("Child")
)

In [20]:
clean_df = clean_df.select(
    "id",
    "FullName",
    "email",
    "age",
    "AgeGroup"
)

In [21]:
clean_df.write \
    .mode("overwrite") \
    .parquet("output/users")

In [22]:
clean_df.write \
    .partitionBy("AgeGroup") \
    .mode("overwrite") \
    .parquet("output/users_partitioned")

In [23]:
clean_df.show()

+---+-----------------+--------------------+---+--------+
| id|         FullName|               email|age|AgeGroup|
+---+-----------------+--------------------+---+--------+
| 14|  Charlotte Lopez|charlotte.lopez@x...| 37|   Adult|
| 11|      Liam Garcia|liam.garcia@x.dum...| 30|   Adult|
|  2| Michael Williams|michael.williams@...| 36|   Adult|
| 12|    Mia Rodriguez|mia.rodriguez@x.d...| 25|   Adult|
|  9|   Ethan Martinez|ethan.martinez@x....| 34|   Adult|
|  6|    Olivia Wilson|olivia.wilson@x.d...| 23|   Adult|
| 15| William Gonzalez|william.gonzalez@...| 33|   Adult|
|  7|  Alexander Jones|alexander.jones@x...| 39|   Adult|
|  1|    Emily Johnson|emily.johnson@x.d...| 29|   Adult|
|  4|      James Davis|james.davis@x.dum...| 46|   Adult|
| 10|Isabella Anderson|isabella.anderson...| 32|   Adult|
| 13|   Noah Hernandez|noah.hernandez@x....| 41|   Adult|
|  3|     Sophia Brown|sophia.brown@x.du...| 43|   Adult|
|  8|       Ava Taylor|ava.taylor@x.dumm...| 28|   Adult|
|  5|      Emm